# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar320250068-prog/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Author:** Omar Hindawy · **Week 2 assignment (ML-03)**

My provisional lane is **Lane 4 — CTR / Engagement Opportunity Scoring** (framed in
`w01_research_question.ipynb`). This notebook maps that lane onto the ML loop from the
`ml-core-foundation-framework` doc: task type → target/proxy → metric → unit of analysis →
why learning beats a fixed rule.

Skills loaded for this task: `skills/framing-ml-problems/SKILL.md` and
`skills/flyrank/flyrank-data/SKILL.md`. The framing text is mine; the cells only compute the
supporting numbers.


## 0. Setup

Locate the repo root so the notebook runs top-to-bottom locally (Jupyter/VS Code) **and** on Colab,
where it clones the repo first.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omar320250068-prog/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    if not os.path.isdir("data/raw"):
        for cand in (r"C:\Users\OMAR HINDAWY\flyrank-ml-internship", os.getcwd()):
            if os.path.isdir(os.path.join(cand, "data", "raw")):
                os.chdir(cand)
                break

import pandas as pd
import numpy as np

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. Ready.")


Working dir: C:\Users\OMAR HINDAWY\flyrank-ml-internship
Starter data found. Ready.


## 1. My lane as an ML task (type)

**Task type: ranking / scoring — with a classification view underneath.**

Lane 4 asks *which visible pages under-capture clicks and should be reviewed first*. That is a
**prioritisation** question, so the natural output is a **ranking**: a page gets a continuous
opportunity **score**, pages are sorted by it, and a human works the top of the list. In the
framework's table, this is **Ranking** ("what should come first?" → a prioritized queue), and the
score is produced by a **classifier** underneath: the model estimates a probability that a page is
"under-capturing," and that probability *is* the ranking score.

So I name it **ranking (score-ordered queue)**, built from a **binary classification** model. I
deliberately do **not** call it plain classification, because the decision is "what order do I show
these in under limited review capacity?", not "is this page good/bad?". A yes/no cut-off with no
ordering would waste the reviewer's scarce time.


In [2]:
# Section 1 backing numbers — why it is a RANKING problem, not a balanced classification problem
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()

print("Total pages:", f"{len(df):,}")
print("Reviewable & visible slice:", f"{len(visible):,}")
print("A reviewer can realistically open a few dozen pages per week -> the decision is ORDER, not pass/fail.")
print("Top-50 style metrics (precision@K) match that decision; raw accuracy would not.")


Total pages: 30,000
Reviewable & visible slice: 12,023
A reviewer can realistically open a few dozen pages per week -> the decision is ORDER, not pass/fail.
Top-50 style metrics (precision@K) match that decision; raw accuracy would not.


## 2. Target or proxy

**The ideal target (what I *really* want): an observed, future-window outcome.**
Something like: *"over the next 30 days, did this page's CTR / position move in a way that shows it
was under-capturing clicks?"* That is the honest, non-circular target — measured **after** the
decision point, from the warehouse's daily facts (features from the prior 90 days → outcome over
the next 30). It is **not** available in the 30k starter slice, which is a single trailing-90-day
snapshot.

**The proxy I can build today (for framing only): a tier-aware under-capture label.**
In the starter slice I approximate "under-capturing" as: **CTR in the bottom quartile for the
page's own position tier** (so I compare a page only to others at a similar rank, per the lane
guide's rule that positions 1–3 always earn more clicks). This is a **defined** proxy, not an
observed future outcome — so I will flag it as a *stand-in* and treat the real target as future
work. Two honest notes:

- The framework's rule is "**the target must be observed, not defined**." This proxy is *defined*,
  so a model trained on it risks learning my rule. I say that out loud now.
- In the starter slice CTR is tiny and coarse (sub-1%, 2 decimals), so an **absolute** gap is
  noisy. The **relative** ratio (`ctr / tier_median`) separates pages far more cleanly, which is
  why the proxy uses a within-tier quartile rather than a fixed percentage-point cutoff.


In [3]:
# Section 2 — what the proxy looks like, and why the relative (within-tier) form is cleaner
visible["pos_bin"] = pd.cut(visible["avg_position"], bins=[0, 3, 10, 20],
                            labels=["top_3", "page_1", "striking"]).astype(str)
tier_median = visible.groupby("pos_bin")["ctr"].transform("median")
visible["ctr_vs_tier"] = visible["ctr"] / tier_median.replace(0, np.nan)

print("Position tiers & their median CTR (the comparison baseline):")
print(visible.groupby("pos_bin")["ctr"].median().round(3).to_string())

print("\nRelative CTR (page / its tier median) quantiles:")
print(visible["ctr_vs_tier"].quantile([.10, .25, .50, .75, .90]).round(3).to_string())

# Proxy: bottom-quartile relative CTR within the tier, with enough volume to matter.
visible["proxy_under_capturing"] = ((visible["ctr_vs_tier"] < 0.50) &
                                   (visible["impressions_90d"] >= 1000)).astype(int)
print("\nProxy target 'under_capturing' base rate:",
      round(visible["proxy_under_capturing"].mean(), 3),
      "->", int(visible["proxy_under_capturing"].sum()), "positives")
print("This is a DEFINED proxy from a snapshot, not a future observed outcome — see the note above.")


Position tiers & their median CTR (the comparison baseline):
pos_bin
page_1      0.24
striking    0.17
top_3       0.20

Relative CTR (page / its tier median) quantiles:
0.10    0.000
0.25    0.471
0.50    1.000
0.75    1.917
0.90    3.371

Proxy target 'under_capturing' base rate: 0.193 -> 2319 positives
This is a DEFINED proxy from a snapshot, not a future observed outcome — see the note above.


## 3. Success metric

**Primary metric: precision@K (K = 20 and K = 50).**

Of the top K pages my ranking puts first, **what fraction are genuinely under-capturing?** That is
exactly the reviewer's question: "if I open the first 50, how many were worth my time?" It matches
the ranked decision, it is computable on any baseline today, and it stays honest when the positive
class is a minority.

**Why not accuracy / ROC-AUC alone?** Accuracy is dominated by the majority class and ignores
order. ROC-AUC measures overall separability but says nothing about whether the *top* of the queue is
good. I will still report **average precision (PR-AUC)** as a secondary view, because it summarises
the whole ranking under class imbalance.

**The baseline that sets the bar:** the starter pipeline's hand rule scored **precision@50 = 0.240**
on the related decline task, and its learned random forest hit **precision@50 = 0.68** (~2.8x). My
lane's target is different, so I must **rebuild** a hand baseline and beat it under the same
**client-holdout** split — not borrow those numbers.

**Cost asymmetry (which error matters):** a false positive wastes reviewer time; a false negative
lets a high-volume page keep leaking clicks. Because the reviewer's capacity is the bottleneck, I
optimise the **top of the queue** (precision@K), and I will read my own top 20 by hand.


In [4]:
# Section 3 — the metric exists and is computable today, on a transparent hand baseline
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y_proxy = visible["proxy_under_capturing"].values

# Transparent hand baseline for THIS lane: relative shortfall x visibility weight.
visible["baseline_opportunity"] = ((1 - visible["ctr_vs_tier"].clip(upper=1))
                                   * np.log1p(visible["impressions_90d"]))
for k in (20, 50):
    print(f"hand baseline precision@{k}: "
          f"{precision_at_k(visible['baseline_opportunity'], y_proxy, k):.3f}")

print("\nThe metric computes on a baseline today (a framing requirement).")
print("Because this baseline and the proxy share the CTR-vs-tier idea, part of the score is")
print("circular by construction — later weeks replace the proxy with a future-window target")
print("and an honest holdout so the comparison means something.")


hand baseline precision@20: 1.000
hand baseline precision@50: 1.000

The metric computes on a baseline today (a framing requirement).
Because this baseline and the proxy share the CTR-vs-tier idea, part of the score is
circular by construction — later weeks replace the proxy with a future-window target
and an honest holdout so the comparison means something.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one page), measured over a trailing 90-day window.**

Grain checks: `content_id` is unique per row (no duplicate pages), and rows come from 32
pseudonymized clients. Predictions for pages of the same client are not independent, so validation
must use a **client-holdout** split (whole clients kept out of training) — the framework's
group-split rule.

Below is the actual lane dataframe: the visible slice, with the columns that will become features,
the proxy target, and the IDs that stay grouping-only.


In [5]:
# Section 4 — the unit of analysis, for real
print("Grain check — is content_id unique per row?",
      "yes" if df["content_id"].is_unique else "NO — investigate")
print(f"One row = one page | rows={len(df):,} | unique pages={df['content_id'].nunique():,} "
      f"| clients={df['client_id'].nunique()}")

feature_cols = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
                "main_intent", "content_type", "content_age_days", "days_since_last_update",
                "sessions_90d", "engagement_rate", "scroll_rate"]

lane_view = visible[["content_id", "client_id"] + feature_cols + ["proxy_under_capturing"]].copy()
print("\nLane dataframe shape:", lane_view.shape)
print("Columns: 2 ID (grouping only) + %d features + 1 proxy target" % len(feature_cols))
lane_view.head(8)


Grain check — is content_id unique per row? yes
One row = one page | rows=30,000 | unique pages=30,000 | clients=32

Lane dataframe shape: (12023, 15)
Columns: 2 ID (grouping only) + 12 features + 1 proxy target


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,main_intent,content_type,content_age_days,days_since_last_update,sessions_90d,engagement_rate,scroll_rate,proxy_under_capturing
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,striking,transactional,keyword article,187,20,17,5.88,4.55,0
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,page_1,commercial,keyword article,463,22,78,1.28,3.45,0
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,page_1,transactional,keyword article,147,20,5,0.00,25.00,1
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,page_1,informational,keyword article,257,104,3,0.00,0.00,0
10,content_d8ee6cc6d642,client_19581e27de,20919,324,1.55,2.2,top_3,commercial,keyword article,329,104,326,6.75,9.55,0
12,content_42fb2cad9ecf,client_6208ef0f77,7228,127,1.76,5.6,page_1,informational,keyword article,124,104,233,3.43,2.61,0
16,content_78bd1d4a1d4d,client_6208ef0f77,13848,21,0.15,8.9,page_1,informational,keyword article,307,104,475,0.21,13.83,0
17,content_761a44afda12,client_19581e27de,9449,7,0.07,7.3,page_1,transactional,keyword article,421,22,118,11.86,15.25,1


In [6]:
# Show the target column's shape and confirm IDs/leak columns are excluded from features
print("Target column 'proxy_under_capturing':")
print(lane_view["proxy_under_capturing"].value_counts().sort_index().to_string())
print(f"positive rate: {lane_view['proxy_under_capturing'].mean():.3f}")

leak_or_ids = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
bad = [c for c in feature_cols if c in leak_or_ids]
print("\nLeakage/ID columns mistakenly used as features:", bad or "none — good")
print("client_id is kept for the grouped split, never fed to the model.")


Target column 'proxy_under_capturing':
proxy_under_capturing
0    9704
1    2319
positive rate: 0.193

Leakage/ID columns mistakenly used as features: none — good
client_id is kept for the grouped split, never fed to the model.


## 5. Why ML beats a fixed rule here

A fixed rule ("flag any page with CTR below X%") fails for four measured reasons in this data:

1. **CTR is not comparable across positions.** A page at position 1 and one at position 15 have
   different normal CTRs, so one global threshold mislabels one of them. That is *why* the target is
   tier-aware — and tiers are only the crudest version of that adjustment.
2. **The obvious signals are individually weak.** `corr(avg_position, ctr) = −0.125` and
   `corr(impressions_90d, ctr) = +0.058` — no single variable orders pages well.
3. **Intent and content type shift the baseline** (median CTR ranges across intents), so the
   "right" amount of clicks depends on several interacting conditions.
4. **Volume decides whether a gap is real or noise**; a low-volume page's CTR wobble is not an
   opportunity.

No single if-statement captures "low CTR *relative to its tier*, at *enough volume*, given its
*intent* and *age*." The signal is real but tangled across several columns — the exact condition the
framework gives for classical ML earning its place over fixed logic. The plain rule stays as the
**baseline to beat**, not as the final answer.


In [7]:
# Section 5 — show the 'tangled' evidence, WITHOUT the column the proxy is built from.
from sklearn.metrics import roc_auc_score

# IMPORTANT honesty note: 'ctr' partly DEFINES the proxy, so including it would just read the
# label back (it scored AUC ~0.94 for exactly that circular reason). We therefore order by the
# OTHER signals a reviewer would actually weigh. Some of these have missing values, so each is
# scored on the rows where it is present (and we report how many that was).
candidate_signals = {
    "avg_position": -visible["avg_position"],
    "impressions_90d": visible["impressions_90d"],
    "content_age_days": visible["content_age_days"],
    "days_since_last_update": visible["days_since_last_update"],
    "engagement_rate": -visible["engagement_rate"],
    "scroll_rate": -visible["scroll_rate"],
}
y_s = pd.Series(y_proxy, index=visible.index)
print("Single-feature ROC-AUC vs the proxy (ctr excluded — it defines the proxy):")
for name, score in candidate_signals.items():
    s = pd.Series(score).astype(float)
    mask = s.notna()
    auc = roc_auc_score(y_s[mask], s[mask])
    print(f"  {name:>24}: AUC = {auc:.3f}  (n={int(mask.sum()):,} of {len(visible):,})")

print("\nNone of the non-defining signals separates well on its own (~0.5-0.6), and some")
print("sit at or below chance. That is the point: no single if-statement on one column orders")
print("these pages. A model that combines several weak signals is what may beat a fixed rule,")
print("and even that has to be re-earned once the proxy becomes a real future-window target.")


Single-feature ROC-AUC vs the proxy (ctr excluded — it defines the proxy):
              avg_position: AUC = 0.471  (n=12,023 of 12,023)
           impressions_90d: AUC = 0.525  (n=12,023 of 12,023)
          content_age_days: AUC = 0.557  (n=12,023 of 12,023)
    days_since_last_update: AUC = 0.528  (n=12,023 of 12,023)
           engagement_rate: AUC = 0.602  (n=12,023 of 12,023)
               scroll_rate: AUC = 0.553  (n=12,020 of 12,023)

None of the non-defining signals separates well on its own (~0.5-0.6), and some
sit at or below chance. That is the point: no single if-statement on one column orders
these pages. A model that combines several weak signals is what may beat a fixed rule,
and even that has to be re-earned once the proxy becomes a real future-window target.


## Self-check

- [x] Every section above is filled — markdown thinking **and** the code that backs it (sections 1–5)
- [x] The notebook runs top to bottom with no errors (executed and saved with outputs)
- [x] No client names, URLs, or private queries anywhere — only the anonymized starter slice, aggregates, and pseudonymized IDs
- [x] Claims use careful words: observed, measured, directional, decision-support — and the proxy target is explicitly flagged as *defined, not observed*
- [x] Committed to my repo under `work/notebooks/`

**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring · **Task type:** ranking/scoring (classification underneath)
**Real target (future work):** observed future-window CTR/position movement · **Proxy today:** tier-relative bottom-quartile under-capture
**Metric:** precision@20 / precision@50 (secondary: PR-AUC) · **Split:** client-holdout
